In [1]:
# imports
from matplotlib import pyplot as plt
import numpy as np
import os
import pandas as pd
import yfinance as yf

from apscheduler.schedulers.blocking import BlockingScheduler
from oandapyV20 import API
import oandapyV20.endpoints.orders as orders
from oandapyV20.contrib.requests import MarketOrderRequest
from oandapyV20.contrib.requests import LimitOrderRequest
from oandapyV20.contrib.requests import TakeProfitDetails, StopLossDetails
from oanda_candles import Pair, Gran, CandleClient

from config import access_token, accountID

In [2]:
# global variables
default_units = 10000
default_instrument = 'USD_JPY'

# earliest 6 mths before no
dataF = yf.download("USDJPY=X", start="2024-11-10", end="2024-12-12", interval="60m")
dataF.iloc[:, :]

# candles
def get_candles(n, order_instrument = default_instrument):

    client = CandleClient(access_token, real=False)

    collector = client.get_collector(Pair.USD_JPY, Gran.M15)

    candles = collector.grab(n)
    return candles

# strategy
def signal_generator(df):
    open = df.Open.iloc[-1]
    close = df.Close.iloc[-1]
    previous_open = df.Open.iloc[-2]
    previous_close = df.Close.iloc[-2]

    # Bearish Pattern
    if (
        open > close
        and previous_open < previous_close
        and close < previous_open
        and open >= previous_close
    ):
        return 1

    # Bullish Pattern
    elif (
        open < close
        and previous_open > previous_close
        and close > previous_open
        and open <= previous_close
    ):
        return 2

    # No clear pattern
    else:
        return 0

    signal = []
    signal.append(0)
    for i in range(1, len(dataF)):
        df = dataF[i - 1 : i + 1]
        signal.append(signal_generator(df))
    # signal_generator(data)
    dataF["signal"] = signal

# execute
def trading_job(order_units=default_units, order_instrument=default_instrument):
    candles = get_candles(3)
    dfstream = pd.DataFrame(columns=["Open", "Close", "High", "Low"])

    i = 0
    for candle in candles:
        dfstream.loc[i, ["Open"]] = float(str(candle.bid.o))
        dfstream.loc[i, ["Close"]] = float(str(candle.bid.c))
        dfstream.loc[i, ["High"]] = float(str(candle.bid.h))
        dfstream.loc[i, ["Low"]] = float(str(candle.bid.l))
        i = i + 1

    dfstream["Open"] = dfstream["Open"].astype(float)
    dfstream["Close"] = dfstream["Close"].astype(float)
    dfstream["High"] = dfstream["High"].astype(float)
    dfstream["Low"] = dfstream["Low"].astype(float)

    signal = signal_generator(dfstream.iloc[:-1, :])  #

    # EXECUTING ORDERS
    client = API(access_token)
    SLTPRatio = 2.0
    previous_candleR = abs(dfstream["High"].iloc[-2] - dfstream["Low"].iloc[-2])
    SLBuy = float(str(candle.bid.o)) - previous_candleR
    SLSell = float(str(candle.bid.o)) + previous_candleR
    TPBuy = float(str(candle.bid.o)) + previous_candleR * SLTPRatio
    TPSell = float(str(candle.bid.o)) - previous_candleR * SLTPRatio
    print(dfstream.iloc[:-1, :])
    print(TPBuy, "  ", SLBuy, "  ", TPSell, "  ", SLSell)
    signal = 2

    # Sell
    if signal == 1:
        mo = MarketOrderRequest(
            instrument=default_instrument,
            units=-order_units,
            takeProfitOnFill=TakeProfitDetails(price=TPSell).data,
            stopLossOnFill=StopLossDetails(price=SLSell).data,
        )
        r = orders.OrderCreate(accountID, data=mo.data)
        rv = client.request(r)
        print(rv)
    # Buy
    elif signal == 2:
        mo = MarketOrderRequest(
            instrument=default_instrument,
            units=order_units,
            takeProfitOnFill=TakeProfitDetails(price=TPBuy).data,
            stopLossOnFill=StopLossDetails(price=SLBuy).data,
        )

        r = orders.OrderCreate(accountID, data=mo.data)
        rv = client.request(r)
        print(rv)

class order(object):
    def __init__(self, 
                instrument="USD_JPY",
                unit = 0,
                en = False,
                p_en = 0,
                tp = False,
                p_tp = 0,
                sl = False,
                p_sl = 0
                ):
        self.instrument = instrument,
        self.unit = unit,
        self.en = en,
        self.p_en = p_en,
        self.tp = tp,
        self.p_tp = p_tp,
        self.sl = sl,
        self.p_sl = p_sl
staged_orders = []

[*********************100%***********************]  1 of 1 completed


In [3]:
def stage_request(
    buyorsell,
    marketorlimit="market"
):
    candles = get_candles(3)
    dfstream = pd.DataFrame(columns=["Open", "Close", "High", "Low"])

    i = 0
    for candle in candles:
        dfstream.loc[i, ["Open"]] = float(str(candle.bid.o))
        dfstream.loc[i, ["Close"]] = float(str(candle.bid.c))
        dfstream.loc[i, ["High"]] = float(str(candle.bid.h))
        dfstream.loc[i, ["Low"]] = float(str(candle.bid.l))
        i = i + 1

    dfstream["Open"] = dfstream["Open"].astype(float)
    dfstream["Close"] = dfstream["Close"].astype(float)
    dfstream["High"] = dfstream["High"].astype(float)
    dfstream["Low"] = dfstream["Low"].astype(float)

    signal = signal_generator(dfstream.iloc[:-1, :])

    # EXECUTING ORDERS
    client = API(access_token)
    SLTPRatio = 2.0
    previous_candleR = abs(dfstream["High"].iloc[-2] - dfstream["Low"].iloc[-2])
    SLBuy = float(str(candle.bid.o)) - previous_candleR
    SLSell = float(str(candle.bid.o)) + previous_candleR
    TPBuy = float(str(candle.bid.o)) + previous_candleR * SLTPRatio
    TPSell = float(str(candle.bid.o)) - previous_candleR * SLTPRatio
    print(dfstream.iloc[:-1, :])
    print(TPBuy, "  ", SLBuy, "  ", TPSell, "  ", SLSell)

    if marketorlimit == "market":
        ordr = MarketOrderRequest(
            instrument="USD_JPY",
            units=1000,
            takeProfitOnFill=TakeProfitDetails(price=TPBuy).data,
            stopLossOnFill=StopLossDetails(price=SLBuy).data,
        )
    else:
        ordr = LimitOrderRequest(instrument="USD_JPY", units=1000, price=1.08)
    print(ordr)
    return


stage_request("market")

      Open    Close     High      Low
0  151.763  151.690  151.787  151.630
1  151.690  151.632  151.692  151.578
151.86    151.518    151.404    151.746
{"_data": {"type": "MARKET", "timeInForce": "FOK", "instrument": "USD_JPY", "units": "1000", "positionFill": "DEFAULT", "clientExtensions": null, "takeProfitOnFill": {"timeInForce": "GTC", "price": "151.86000"}, "stopLossOnFill": {"timeInForce": "GTC", "price": "151.51800"}, "trailingStopLossOnFill": null, "tradeClientExtensions": null}}


In [4]:
# Stage order
# staged_orders.append(order(1, p_tp=100))
# staged_orders.append(order(2, p_en=50))
staged_orders.append(
    order(
        instrument="USD_JPY",  # forex pair
        unit=10000,  # trade unit
        en=False,  # entry order (True) / market order (false)
        p_en=0,  # entry price
        tp=False,  # take profit set (True) / not set(False)
        p_tp=0,  # take profit price
        sl=False,  # stop loss set (True) / not set(False)
        p_sl=0,  # stop loss price
    )
)

In [5]:
# show staged
print(pd.DataFrame([t.__dict__ for t in staged_orders]))

   instrument      unit        en  p_en        tp  p_tp        sl  p_sl
0  (USD_JPY,)  (10000,)  (False,)  (0,)  (False,)  (0,)  (False,)     0


In [6]:
client = API(access_token)
r = orders.OrderList(accountID)
client.request(r)
print(r.response)
print(r.response['orders'])
for order in r.response['orders']:
  print(order)
  
# sort orders into 

{'orders': [{'id': '43', 'createTime': '2025-02-08T12:03:56.369987319Z', 'type': 'LIMIT', 'instrument': 'USD_JPY', 'units': '-1', 'timeInForce': 'GTD', 'price': '153.315', 'gtdTime': '2025-02-15T12:03:53.000000000Z', 'triggerCondition': 'DEFAULT', 'partialFill': 'DEFAULT_FILL', 'positionFill': 'DEFAULT', 'state': 'PENDING'}, {'id': '42', 'createTime': '2025-02-08T12:03:47.648089314Z', 'type': 'LIMIT', 'instrument': 'USD_JPY', 'units': '-1', 'timeInForce': 'GTD', 'price': '153.781', 'gtdTime': '2025-02-15T12:03:45.000000000Z', 'triggerCondition': 'DEFAULT', 'partialFill': 'DEFAULT_FILL', 'positionFill': 'DEFAULT', 'state': 'PENDING'}, {'id': '41', 'createTime': '2025-02-08T12:03:38.399425147Z', 'type': 'LIMIT', 'instrument': 'USD_JPY', 'units': '-1', 'timeInForce': 'GTD', 'price': '152.748', 'gtdTime': '2025-02-15T12:03:29.000000000Z', 'triggerCondition': 'DEFAULT', 'partialFill': 'DEFAULT_FILL', 'positionFill': 'DEFAULT', 'state': 'PENDING'}, {'id': '33', 'createTime': '2025-02-03T21:2

In [7]:
# Staged run


# single run
# trading_job()

# recurring run
# scheduler = BlockingScheduler()
# scheduler.add_job(trading_job, 'cron', day_of_week='mon-fri', hour='00-23', minute='1,16,31,46', start_date='2022-01-12 12:00:00', timezone='America/Chicago')
# scheduler.start()